In [ ]:
# Parquet Write Example with PySpark and DataFrames

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ParquetWriteExample").getOrCreate()

data = [("Alice", 25), ("Bob", 30)]

df = spark.createDataFrame(data, ["name", "age"])

df.write.parquet("output.parquet", mode="overwrite")

print("""
Output in output.parquet/part-00000-*.parquet:
(binary Parquet data with schema: name: string, age: int)
""")

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/01 10:29:28 WARN Utils: Your hostname, AsusLaptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/01 10:29:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 10:29:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/01 10:29:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.



Output in output.parquet/part-00000-*.parquet:
(binary Parquet data with schema: name: string, age: int)



In [5]:
# Example with Parquet compression and partitioning

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ParquetParams").getOrCreate()

data = [("Alice", 25, "East"), ("Bob", 30, "West")]

df = spark.createDataFrame(data, ["name", "age", "region"])

                                                    # "snappy" has a balance of speed and size
                                                    # "gzip" has better compression, but is slower
df.write.parquet("output.parquet", mode="overwrite", compression="gzip", partitionBy=["region"])

print("""
Output structure:
output.parquet/region=East/part-00000-*.parquet.gz
output.parquet/region=West/part-00000-*.parquet.gz
""")


26/07/01 10:45:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.



Output structure:
output.parquet/region=East/part-00000-*.parquet.gz
output.parquet/region=West/part-00000-*.parquet.gz



In [ ]:
# Read the partitioned Parquet dataset back into Spark

parquet_df = spark.read.parquet("output.parquet")

print("\nSchema from Parquet metadata:")
parquet_df.printSchema()

print("\nContents:")
parquet_df.show()

# --------------------------------------------------------------------------------------------------------------
# Filter on the partition column
# Spark only scans the matching partition (partition pruning, skip unrelated directories)
# --------------------------------------------------------------------------------------------------------------

east_df = parquet_df.filter(parquet_df.region == "East")

print("\nOnly East records:")
east_df.show()

spark.stop()


Schema from Parquet metadata:
root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- region: string (nullable = true)


Contents:
+-----+---+------+
| name|age|region|
+-----+---+------+
|Alice| 25|  East|
|  Bob| 30|  West|
+-----+---+------+


Only East records:
+-----+---+------+
| name|age|region|
+-----+---+------+
|Alice| 25|  East|
+-----+---+------+

